# Building a circuit from scratch (and the netlist text format)

Two equivalent ways to define a circuit: the **Python `add_*` API** and a
**netlist text file**. We build a **fluxonium** (a Josephson junction shunted by
a large inductor) both ways and confirm they give the same Hamiltonian.

In [ ]:
import numpy as np
from fluxcharge import Circuit

# add_<element>(edge_name, tail_node, head_node, value)
fx = Circuit()
fx.add_josephson("e1", "v1", "v2", EJ="E_J")
fx.add_inductor("e2", "v1", "v2", L="L")
fx.add_capacitor("e3", "v1", "v2", C="C")
# loops are optional: if you declare none, fluxcharge infers the planar faces.
res = fx.hamiltonian()
res.H

## Gauge is optional

One node flux and one loop charge are always redundant; the reduction finds and
removes them itself, so `hamiltonian()` completes for **any** ground/open choice
or none. Naming a `ground` node or `open_loops` only fixes *which* representative
survives — it never changes whether the reduction completes, nor the spectrum.

In [ ]:
res_a = fx.hamiltonian()                                  # no gauge specified
res_b = fx.hamiltonian(ground="v1", open_loops="f3")      # explicit gauge
print("no gauge :", res_a.H)
print("gauged   :", res_b.H)
print("both complete:", res_a.complete, res_b.complete)

## The netlist text format

The same circuit as text. Format:

```
TYPE name node1 node2 [value]      # C / L / J / QPS
gyrator e1 a b  e2 a b  [ratio]
loop name +e1 -e2 ...              # signed edge list (a face)
ground node                        # optional global-flux gauge
open   loop                        # optional global-charge gauge
flux   loop [value]                # external flux bias
offset node [value]                # offset / gate charge
title  ...
```

In [ ]:
from fluxcharge import from_netlist, to_netlist

netlist = '''
title Fluxonium
J e1 v1 v2 E_J
L e2 v1 v2 L
C e3 v1 v2 C
'''
ckt = from_netlist(netlist)
print("from netlist:", ckt.hamiltonian().H)

# serialize any circuit back to a netlist (round-trips)
print("\n--- to_netlist(fx) ---")
print(to_netlist(fx))

## Draw it

`draw_schematic` renders a planar schematic; pass `file=` to save a PNG. The
outer face is auto-detected from the planar embedding.

In [ ]:
from fluxcharge import draw_schematic
draw_schematic(fx, file="fluxonium_schematic.png")
from IPython.display import Image          # display the saved PNG inline
Image("fluxonium_schematic.png")

**Takeaway.** Build in Python or in a netlist file — they're interchangeable
(`from_netlist` / `to_netlist`). Loops and gauge are optional; fluxcharge infers
the faces and removes the redundant coordinates on its own.